# Chapter 2: Understanding and Preparing Video Data for Model Training

## Companion Notebook

**Learning Objective:** In this chapter, you'll learn how to identify different types of video data, understand their unique challenges, and use practical tools to build scalable data pipelines for AI.

This notebook accompanies Chapter 2 of the book and provides all the hands-on code examples referenced in the text. Follow along with the book to understand the context and theory behind each code cell.

## Setting Up the Workspace

The first step in our preprocessing pipeline is to create an organized workspace for raw files, processed outputs, and metadata.

In [2]:
from pathlib import Path

workspace = Path("/content/video_pipeline_demo") # Define workspace in the global scope
subdirs = [
    "raw", "compressed", "frames", "scenes",
    "annotations", "metadata", "analysis",
    "checkpoints", "pet_videos"
]

print("🎬 Setting up your video pipeline workspace...")
workspace.mkdir(exist_ok=True)

for folder in subdirs:
    (workspace / folder).mkdir(exist_ok=True)
    print(f"  📂 {folder}/")

print("\n✅ Workspace ready!")
print(f"  Working directory: {workspace}")

🎬 Setting up your video pipeline workspace...
  📂 raw/
  📂 compressed/
  📂 frames/
  📂 scenes/
  📂 annotations/
  📂 metadata/
  📂 analysis/
  📂 checkpoints/
  📂 pet_videos/

✅ Workspace ready!
  Working directory: /content/video_pipeline_demo


## Loading the Dataset

With the workspace folders in place, the next step is to load a sample dataset. We'll use a curated gallery of pet videos as our demonstration dataset.

In [3]:
# Mock Gallery class for demonstration
# In production, this would load actual pet videos from Chapter 1
class Gallery:
    def __init__(self):
        self.videos = []

    def download_all_videos(self):
        """Simulate downloading pet videos"""
        # In real implementation, this would download from a repository
        import requests
        from pathlib import Path

        # Sample video URLs (using freely available test videos)
        sample_urls = [
            ("https://ia800207.us.archive.org/15/items/BigBuckBunny_124/Content/big_buck_bunny_720p_surround.mp4", "wizard_cat_raw.mp4"),
            ("https://archive.org/download/TheSpiritOf43_56/The_Spirit_of__43_512kb.mp4", "playful_dog.mp4"),
        ]

        video_dir = workspace / "pet_videos"
        video_dir.mkdir(exist_ok=True)

        for url, filename in sample_urls:
            filepath = video_dir / filename
            if not filepath.exists():
                try:
                    print(f"Downloading {filename}...")
                    response = requests.get(url)
                    filepath.write_bytes(response.content)
                    self.videos.append(filepath)
                except Exception as e:
                    print(f"Could not download {filename}: {e}")
            else:
                self.videos.append(filepath)

        return self.videos

    def get_video_stats(self):
        """Get statistics about downloaded videos"""
        import pandas as pd
        import cv2

        stats = []
        for video_path in self.videos:
            cap = cv2.VideoCapture(str(video_path))
            stats.append({
                'filename': video_path.name,
                'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
                'fps': cap.get(cv2.CAP_PROP_FPS),
                'frames': int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            })
            cap.release()

        return pd.DataFrame(stats)

    def display_gallery(self):
        """Display video thumbnails"""
        print("[Gallery visualization would appear here]")
        for video in self.videos:
            print(f"  🎥 {video.name}")

# Import gallery class from notebook
gallery = Gallery()

# Download all pet videos
videos = gallery.download_all_videos()

# Display video statistics
print("\n📊 Video Statistics")
print("="*60)
stats_df = gallery.get_video_stats()
display(stats_df)


📊 Video Statistics


,filename,width,height,fps,frames
0,wizard_cat_raw.mp4,640,360,24.000000,14315
1,playful_dog.mp4,320,240,29.922822,10181


In [ ]:
print("\n🖼️ Pet Video Thumbnail Gallery")
print("="*60)
print("Here's your collection of amazing AI-generated pet videos!\n")
gallery.display_gallery()

## Types of Video Data

### Raw Video Data

Raw video refers to untouched footage captured directly from devices. Let's analyze the basic properties of raw video files.

In [4]:
import cv2

def analyze_raw_video(video_path):
    """Analyze basic properties of raw video data"""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open: {video_path}")

    # Core properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    # Derived metrics
    duration_seconds = frame_count / fps if fps > 0 else 0
    bytes_per_frame = width * height * 3  # RGB
    estimated_size_gb = (bytes_per_frame * frame_count) / (1024 ** 3)

    return {
        "width": width, "height": height, "fps": fps,
        "frame_count": frame_count, "duration_seconds": duration_seconds,
        "estimated_size_gb": estimated_size_gb,
    }

# Example: Analyzing a sample raw video
if videos:
    props = analyze_raw_video(videos[0])
    print(f"Resolution: {props['width']}x{props['height']} @ {props['fps']} fps")
    print(f"Frames: {props['frame_count']} | Duration: {props['duration_seconds']:.2f} sec")
    print(f"Estimated raw size: {props['estimated_size_gb']:.2f} GB")

Resolution: 640x360 @ 24.0 fps
Frames: 14315 | Duration: 596.46 sec
Estimated raw size: 9.21 GB


### Compressed Video Data

Most real-world video workflows rely on compression. Let's compress a video using FFmpeg and compare file sizes.

In [5]:
def compress_video(input_path, output_path, codec="libx264", crf=23):
    """Compress video using FFmpeg and compare sizes."""
    import subprocess
    import os

    cmd = [
        "ffmpeg", "-y", "-i", str(input_path),
        "-c:v", codec, "-crf", str(crf),
        str(output_path)
    ]

    try:
        subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

        # Compare file sizes
        original_size = os.path.getsize(input_path) / (1024 ** 2)
        compressed_size = os.path.getsize(output_path) / (1024 ** 2)

        return {
            "original_size_mb": original_size,
            "compressed_size_mb": compressed_size,
            "compression_ratio": original_size / compressed_size if compressed_size > 0 else 0,
        }
    except (subprocess.CalledProcessError, FileNotFoundError) as e:
        print(f"Note: FFmpeg not available or error occurred: {e}")
        print("Install FFmpeg to enable compression features")
        return None

if videos:
    output_path = workspace / "compressed" / "wizard_cat_compressed.mp4"
    stats = compress_video(videos[0], output_path)
    if stats:
        print(f"Original size   : {stats['original_size_mb']:.2f} MB")
        print(f"Compressed size : {stats['compressed_size_mb']:.2f} MB")
        print(f"Ratio          : {stats['compression_ratio']:.2f}x smaller")

KeyboardInterrupt: 

### Annotated Video Data

For supervised learning, video must often be enriched with annotations. Here's how to work with annotated clips.

In [6]:
def extract_annotated_clips(annotations, action_type):
    """Extract video segments containing a specific action"""
    clips = []
    for ann in annotations:
        if ann.get("action") == action_type:
            clips.append({
                "start_frame": ann.get("start_frame", ann["frame_id"]),
                "end_frame": ann.get("end_frame", ann["frame_id"]),
                "confidence": ann.get("confidence", 1.0)
            })
    return clips

# Example: loading JSON annotations
annotations = [
    {"frame_id": 40, "action": "jump", "confidence": 0.95},
    {"frame_id": 42, "action": "jump", "confidence": 0.92},
]

jump_clips = extract_annotated_clips(annotations, "jump")
print(jump_clips)

[{'start_frame': 40, 'end_frame': 40, 'confidence': 0.95}, {'start_frame': 42, 'end_frame': 42, 'confidence': 0.92}]


### Streaming Video Data

Streaming video is central to latency-sensitive applications. Let's simulate streaming by reading frames sequentially.

In [7]:
def stream_video(source, max_frames=3):
    """Simulate streaming video by reading frames one by one."""
    import cv2

    cap = cv2.VideoCapture(str(source))
    frame_count = 0

    while cap.isOpened() and frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        print(f"Frame {frame_count}: shape={frame.shape}")
        frame_count += 1

    cap.release()

# Example: stream compressed video
if videos:
    stream_video(videos[0])

Frame 0: shape=(360, 640, 3)
Frame 1: shape=(360, 640, 3)
Frame 2: shape=(360, 640, 3)


## Building the Pipeline: Core Components

### Quality Assessment with OpenCV

In [8]:
import numpy as np

class OpenCVQualityAssessor:
    """Assess video quality using OpenCV metrics."""

    def __init__(self, sharpness_threshold=100):
        self.sharpness_threshold = sharpness_threshold

    def assess(self, video_path, sample_frames=5):
        """Assess video quality by sampling frames."""
        cap = cv2.VideoCapture(str(video_path))

        # Get video properties
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = total_frames / fps if fps > 0 else 0

        # Sample frames for quality assessment
        frame_indices = np.linspace(0, total_frames-1, sample_frames, dtype=int)
        sharpness_scores = []
        motion_scores = []
        prev_gray = None

        for idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                continue

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            # Measure sharpness using Laplacian variance
            laplacian = cv2.Laplacian(gray, cv2.CV_64F)
            sharpness = laplacian.var()
            sharpness_scores.append(sharpness)

            # Measure motion using optical flow
            if prev_gray is not None:
                flow = cv2.calcOpticalFlowFarneback(
                    prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0
                )
                motion = np.mean(np.sqrt(flow[...,0]**2 + flow[...,1]**2))
                motion_scores.append(motion)

            prev_gray = gray

        cap.release()

        avg_sharpness = np.mean(sharpness_scores) if sharpness_scores else 0
        avg_motion = np.mean(motion_scores) if motion_scores else 0

        return {
            'resolution': f"{width}x{height}",
            'fps': fps,
            'duration': duration,
            'avg_sharpness': avg_sharpness,
            'avg_motion': avg_motion,
            'is_sharp': avg_sharpness > self.sharpness_threshold
        }

# Example usage
assessor = OpenCVQualityAssessor()
if videos:
    quality_report = assessor.assess(videos[0])
    print("Quality Assessment Report:")
    for key, value in quality_report.items():
        print(f"  {key}: {value}")

Quality Assessment Report:
  resolution: 640x360
  fps: 24.0
  duration: 596.4583333333334
  avg_sharpness: 109.92613131373275
  avg_motion: 10.356605529785156
  is_sharp: True


### Video Export with FFmpeg

In [9]:
class FFmpegExporter:
    """Export videos using FFmpeg with various settings."""

    @staticmethod
    def trim(video_path, start_time, duration, out_path):
        """Extract a segment from a video."""
        import subprocess

        cmd = [
            "ffmpeg", "-y",
            "-ss", str(start_time),
            "-i", str(video_path),
            "-t", str(duration),
            "-c:v", "libx264",
            "-c:a", "copy",
            str(out_path)
        ]

        try:
            subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            print(f"✓ Trimmed video saved to {out_path}")
            return True
        except (subprocess.CalledProcessError, FileNotFoundError):
            print("Note: FFmpeg not available")
            return False

    @staticmethod
    def transcode_resize(video_path, out_path, size=(512, 512), crf=23, preset="medium"):
        """Resize and transcode video."""
        import subprocess

        w, h = size
        vf = f"scale={w}:{h}:force_original_aspect_ratio=decrease"

        cmd = [
            "ffmpeg", "-y",
            "-i", str(video_path),
            "-vf", vf,
            "-c:v", "libx264", "-crf", str(crf), "-preset", preset,
            "-c:a", "copy",
            str(out_path)
        ]

        try:
            subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            print(f"✓ Resized video saved to {out_path}")
            return True
        except (subprocess.CalledProcessError, FileNotFoundError):
            print("Note: FFmpeg not available")
            return False

# Example: Trim a video segment
exporter = FFmpegExporter()
if videos:
    trimmed_path = workspace / "compressed" / "trimmed_clip.mp4"
    exporter.trim(videos[0], start_time=1, duration=2, out_path=trimmed_path)

✓ Trimmed video saved to /content/video_pipeline_demo/compressed/trimmed_clip.mp4


### Scene Detection

In [10]:
class SceneDetector:
    """Detect scene boundaries in videos."""

    def __init__(self, threshold=30.0):
        self.threshold = threshold

    def detect_scenes(self, video_path):
        """Detect scene changes using frame differences."""
        cap = cv2.VideoCapture(str(video_path))
        scenes = []
        frame_num = 0
        prev_gray = None

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            if prev_gray is not None:
                # Calculate frame difference
                diff = cv2.absdiff(prev_gray, gray)
                mean_diff = np.mean(diff)

                # Detect scene change
                if mean_diff > self.threshold:
                    scenes.append({
                        'frame': frame_num,
                        'time': frame_num / cap.get(cv2.CAP_PROP_FPS),
                        'intensity': mean_diff
                    })

            prev_gray = gray
            frame_num += 1

        cap.release()
        return scenes

# Example usage
detector = SceneDetector()
if videos:
    scenes = detector.detect_scenes(videos[0])
    print(f"Detected {len(scenes)} scene changes")
    for i, scene in enumerate(scenes[:3]):  # Show first 3
        print(f"  Scene {i+1}: Frame {scene['frame']}, Time {scene['time']:.2f}s")

Detected 185 scene changes
  Scene 1: Frame 285, Time 11.88s
  Scene 2: Frame 378, Time 15.75s
  Scene 3: Frame 553, Time 23.04s


## Scalable Pipeline Components

### Filtering Pipeline

In [12]:
class ScalableFilteringPipeline:
    """Scalable pipeline for filtering video datasets."""

    def __init__(self, min_resolution=128, min_duration=2.0, max_duration=60.0):
        self.min_resolution = min_resolution
        self.min_duration = min_duration
        self.max_duration = max_duration
        self.assessor = OpenCVQualityAssessor()

    def process_video(self, video_path):
        """Process a single video through the filtering pipeline."""
        # Assess quality
        quality = self.assessor.assess(video_path)

        # Apply filters
        passed = True
        reasons = []

        # Resolution filter
        width, height = quality['resolution'].split('x')
        if min(int(width), int(height)) < self.min_resolution:
            passed = False
            reasons.append(f"Resolution too low: {quality['resolution']}")

        # Duration filter
        if quality['duration'] < self.min_duration:
            passed = False
            reasons.append(f"Too short: {quality['duration']:.1f}s")
        elif quality['duration'] > self.max_duration:
            passed = False
            reasons.append(f"Too long: {quality['duration']:.1f}s")

        # Quality filter
        if not quality['is_sharp']:
            passed = False
            reasons.append("Video is too blurry")

        return {
            'video_path': video_path,
            'passed': passed,
            'quality': quality,
            'rejection_reasons': reasons
        }

    def batch_process(self, video_paths):
        """Process multiple videos."""
        results = []
        for path in video_paths:
            result = self.process_video(path)
            results.append(result)

            status = "✓ Passed" if result['passed'] else "✗ Failed"
            print(f"{path.name}: {status}")
            if result['rejection_reasons']:
                for reason in result['rejection_reasons']:
                    print(f"    - {reason}")

        passed = sum(1 for r in results if r['passed'])
        print(f"\nSummary: {passed}/{len(results)} videos passed filtering")
        return results

# Example: Run filtering pipeline
pipeline = ScalableFilteringPipeline()
if videos:
    results = pipeline.batch_process(videos)

wizard_cat_raw.mp4: ✗ Failed
    - Too long: 596.5s
playful_dog.mp4: ✗ Failed
    - Too long: 340.2s

Summary: 0/2 videos passed filtering


### Annotation Pipeline

In [ ]:
class ScalableAnnotationPipeline:
    """Pipeline for generating video annotations."""

    def __init__(self):
        self.detector = SceneDetector()

    def generate_annotations(self, video_path):
        """Generate comprehensive annotations for a video."""
        # Basic metadata
        cap = cv2.VideoCapture(str(video_path))
        metadata = {
            'filename': video_path.name,
            'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
            'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
            'fps': cap.get(cv2.CAP_PROP_FPS),
            'frames': int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        }
        cap.release()

        # Scene detection
        scenes = self.detector.detect_scenes(video_path)

        # Mock object detection (in practice, use YOLO or similar)
        mock_objects = [
            {'frame': 10, 'object': 'cat', 'confidence': 0.95},
            {'frame': 50, 'object': 'toy', 'confidence': 0.87}
        ]

        # Mock actions (in practice, use action recognition model)
        mock_actions = [
            {'start': 0, 'end': 30, 'action': 'playing'},
            {'start': 31, 'end': 60, 'action': 'resting'}
        ]

        return {
            'metadata': metadata,
            'scenes': scenes,
            'objects': mock_objects,
            'actions': mock_actions
        }

    def save_annotations(self, annotations, output_path):
        """Save annotations to JSON file."""
        import json

        with open(output_path, 'w') as f:
            json.dump(annotations, f, indent=2)
        print(f"✓ Annotations saved to {output_path}")

# Example usage
annotator = ScalableAnnotationPipeline()
if videos:
    annotations = annotator.generate_annotations(videos[0])

    # Save annotations
    anno_path = workspace / "annotations" / f"{videos[0].stem}_annotations.json"
    annotator.save_annotations(annotations, anno_path)

    print("\nGenerated Annotations:")
    print(f"  Scenes detected: {len(annotations['scenes'])}")
    print(f"  Objects found: {len(annotations['objects'])}")
    print(f"  Actions identified: {len(annotations['actions'])}")

## Advanced Techniques

### Adaptive Frame Extraction

In [ ]:
class AdaptiveFrameExtractor:
    """Extract frames adaptively based on content changes."""

    def extract_frames(self, video_path, min_frames=5, max_frames=30, threshold=20):
        """Extract frames when significant changes occur."""
        cap = cv2.VideoCapture(str(video_path))
        frames = []
        frame_indices = []
        prev_gray = None
        frame_num = 0

        while len(frames) < max_frames:
            ret, frame = cap.read()
            if not ret:
                break

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            # Always include first frame
            if prev_gray is None:
                frames.append(frame)
                frame_indices.append(frame_num)
            else:
                # Calculate difference
                diff = cv2.absdiff(prev_gray, gray)
                mean_diff = np.mean(diff)

                # Extract frame if significant change
                if mean_diff > threshold:
                    frames.append(frame)
                    frame_indices.append(frame_num)

            prev_gray = gray
            frame_num += 1

        cap.release()

        # Ensure minimum frames
        if len(frames) < min_frames:
            # Add evenly spaced frames
            cap = cv2.VideoCapture(str(video_path))
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

            for i in np.linspace(0, total_frames-1, min_frames - len(frames), dtype=int):
                cap.set(cv2.CAP_PROP_POS_FRAMES, i)
                ret, frame = cap.read()
                if ret:
                    frames.append(frame)
                    frame_indices.append(i)

            cap.release()

        return frames, frame_indices

    def save_frames(self, frames, output_dir, prefix="frame"):
        """Save extracted frames to disk."""
        output_dir = Path(output_dir)
        output_dir.mkdir(exist_ok=True)

        for i, frame in enumerate(frames):
            filename = output_dir / f"{prefix}_{i:04d}.jpg"
            cv2.imwrite(str(filename), frame)

        print(f"✓ Saved {len(frames)} frames to {output_dir}")

# Example usage
extractor = AdaptiveFrameExtractor()
if videos:
    frames, indices = extractor.extract_frames(videos[0])
    print(f"Extracted {len(frames)} frames at indices: {indices[:5]}...")

    # Save frames
    frames_dir = workspace / "frames" / videos[0].stem
    extractor.save_frames(frames, frames_dir)

### Concept Balancing

In [ ]:
class ConceptBalancer:
    """Balance dataset based on concept distribution."""

    def __init__(self, target_samples_per_concept=100):
        self.target_samples = target_samples_per_concept

    def analyze_distribution(self, annotations_list):
        """Analyze concept distribution in dataset."""
        concept_counts = {}

        for anno in annotations_list:
            # Count objects
            for obj in anno.get('objects', []):
                concept = obj['object']
                concept_counts[concept] = concept_counts.get(concept, 0) + 1

            # Count actions
            for action in anno.get('actions', []):
                concept = action['action']
                concept_counts[concept] = concept_counts.get(concept, 0) + 1

        return concept_counts

    def balance_dataset(self, videos_with_annotations):
        """Balance dataset by sampling videos per concept."""
        # Group videos by concept
        concept_videos = {}

        for video, anno in videos_with_annotations:
            for obj in anno.get('objects', []):
                concept = obj['object']
                if concept not in concept_videos:
                    concept_videos[concept] = []
                concept_videos[concept].append(video)

        # Sample videos for balance
        balanced_dataset = set()

        for concept, videos in concept_videos.items():
            # Sample up to target number
            sample_size = min(len(videos), self.target_samples)
            sampled = np.random.choice(videos, sample_size, replace=False)
            balanced_dataset.update(sampled)

        return list(balanced_dataset)

# Example usage
balancer = ConceptBalancer(target_samples_per_concept=50)

# Mock dataset with annotations
mock_dataset = [
    ('video1.mp4', {'objects': [{'object': 'cat'}, {'object': 'toy'}]}),
    ('video2.mp4', {'objects': [{'object': 'dog'}]}),
    ('video3.mp4', {'objects': [{'object': 'cat'}]}),
]

# Analyze distribution
distribution = balancer.analyze_distribution([anno for _, anno in mock_dataset])
print("Concept Distribution:")
for concept, count in distribution.items():
    print(f"  {concept}: {count}")

# Balance dataset
balanced = balancer.balance_dataset(mock_dataset)
print(f"\nBalanced dataset contains {len(balanced)} videos")

## Data Augmentation

### Temporal Augmentation

In [ ]:
def temporal_augmentation(video_frames, speed_factor=1.5):
    """Apply temporal augmentation by changing playback speed."""
    if speed_factor == 1.0:
        return video_frames

    # Sample frames at different rate
    indices = np.arange(0, len(video_frames), speed_factor)
    indices = indices.astype(int)
    indices = indices[indices < len(video_frames)]

    augmented = [video_frames[i] for i in indices]
    return augmented

# Example
print(f"Original video: 100 frames")
print(f"Speed 1.5x: {len(temporal_augmentation(list(range(100)), 1.5))} frames")
print(f"Speed 0.5x: {len(temporal_augmentation(list(range(100)), 0.5))} frames")

### Spatial Augmentation

In [ ]:
def spatial_augmentation(frame, flip=True, rotate=True, brightness=1.0):
    """Apply spatial augmentations to a frame."""
    augmented = frame.copy()

    # Horizontal flip
    if flip and np.random.random() > 0.5:
        augmented = cv2.flip(augmented, 1)

    # Random rotation
    if rotate:
        angle = np.random.uniform(-10, 10)
        h, w = augmented.shape[:2]
        M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1)
        augmented = cv2.warpAffine(augmented, M, (w, h))

    # Brightness adjustment
    if brightness != 1.0:
        augmented = cv2.convertScaleAbs(augmented, alpha=brightness, beta=0)

    return augmented

print("Spatial augmentations available:")
print("  - Horizontal flip")
print("  - Random rotation (-10° to +10°)")
print("  - Brightness adjustment")

## Putting It All Together: End-to-End Pipeline

Let's combine all components into a complete preprocessing pipeline that mimics production systems at a smaller scale.

In [ ]:
def run_complete_pipeline(video_paths, workspace):
    """Run the complete video preprocessing pipeline."""

    print("🚀 Starting Complete Video Processing Pipeline")
    print("="*60)

    # Stage 1: Quality Assessment & Filtering
    print("\n📊 Stage 1: Quality Assessment & Filtering")
    filter_pipeline = ScalableFilteringPipeline()
    filter_results = filter_pipeline.batch_process(video_paths)

    # Get videos that passed filtering
    passed_videos = [r['video_path'] for r in filter_results if r['passed']]

    if not passed_videos:
        print("No videos passed filtering. Adjusting criteria...")
        # Use all videos for demonstration
        passed_videos = video_paths

    # Stage 2: Scene Detection & Frame Extraction
    print("\n🎬 Stage 2: Scene Detection & Frame Extraction")
    extractor = AdaptiveFrameExtractor()

    for video_path in passed_videos:
        frames, indices = extractor.extract_frames(video_path, min_frames=3, max_frames=10)
        frames_dir = workspace / "frames" / video_path.stem
        extractor.save_frames(frames[:5], frames_dir)  # Save first 5 frames

    # Stage 3: Annotation Generation
    print("\n🏷️ Stage 3: Annotation Generation")
    annotator = ScalableAnnotationPipeline()
    all_annotations = []

    for video_path in passed_videos:
        annotations = annotator.generate_annotations(video_path)
        all_annotations.append(annotations)

        # Save annotations
        anno_path = workspace / "annotations" / f"{video_path.stem}.json"
        annotator.save_annotations(annotations, anno_path)

    # Stage 4: Dataset Balancing
    print("\n⚖️ Stage 4: Dataset Balancing")
    balancer = ConceptBalancer()
    distribution = balancer.analyze_distribution(all_annotations)

    print("Concept distribution in dataset:")
    for concept, count in distribution.items():
        print(f"  {concept}: {count}")

    # Stage 5: Summary Statistics
    print("\n📈 Pipeline Summary")
    print("="*60)
    print(f"Total videos processed: {len(video_paths)}")
    print(f"Videos passed filtering: {len(passed_videos)}")
    print(f"Total frames extracted: {sum(len(list((workspace/'frames'/v.stem).glob('*.jpg'))) for v in passed_videos if (workspace/'frames'/v.stem).exists())}")
    print(f"Annotations generated: {len(all_annotations)}")

    return {
        'filter_results': filter_results,
        'annotations': all_annotations,
        'passed_videos': passed_videos
    }

# Run the complete pipeline
if videos:
    pipeline_results = run_complete_pipeline(videos, workspace)
    print("\n✅ Pipeline completed successfully!")

## Summary

In this chapter, you learned how to approach video data not just as raw footage, but as the essential foundation for generative AI. You have:

- **Understood different types of video data** (raw, compressed, annotated, streaming)
- **Built quality assessment tools** using OpenCV
- **Implemented filtering pipelines** to ensure data quality
- **Created annotation systems** for supervised learning
- **Developed frame extraction** strategies
- **Applied augmentation techniques** to enrich datasets
- **Combined everything** into a scalable pipeline

The techniques you've learned here form the foundation for preparing video datasets at any scale, from small experiments to production systems processing millions of videos.

### Next Steps

In the next chapter, we'll take the preprocessed datasets you've created here and use them as inputs for training generative models, exploring how clean, well-prepared data becomes the fuel for innovation in video generation with AI.